## 📝 Hipótesis pre-registradas para K-Means
Basado en diagnóstico estadístico y matriz inter-tasas [0.62-0.82], se espera que K-Means identifique:
1. 🔴 Crisis VIF intergeneracional (alta VIF NNA + VIF adultas)
2. 🟠 Crisis sexual dominante (alta sexual NNA + sexual adultas)
3. 🟡 Riesgo moderado balanceado (tasas medias, sin dominancia)
4. 🟢 Bajo reporte/validar (tasas bajas, posible subregistro)

In [1]:
# ✅ CELDA 1: IMPORTS + CONFIG + CARGA DE DATOS
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score
import joblib
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

with open(PROJECT_ROOT / "config" / "ml_config.json", 'r', encoding='utf-8') as f:
    ml_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "base_config.json", 'r', encoding='utf-8') as f:
    base_cfg = json.load(f)

disp = base_cfg.get('display_config', {})
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO')
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')

clustering_df = pd.read_parquet(PROJECT_ROOT / "data/master" / "tabla_clustering.parquet")
print(f"{TITLE.format(n=1, module='IMPORTS Y CONFIGURACIÓN')}")
print(f"{IND}📦 ml_config.json cargado | Modelo: {ml_cfg['model']}")
print(f"{IND}📊 tabla_clustering.parquet: {len(clustering_df)} municipios")
print(f"{IND}🔍 Granularidad: 1 fila por municipio (promedio 2018-2025)")
print(SEP)

✅ CELDA 1 - IMPORTS Y CONFIGURACIÓN EJECUTADO
   📦 ml_config.json cargado | Modelo: KMeans
   📊 tabla_clustering.parquet: 179 municipios
   🔍 Granularidad: 1 fila por municipio (promedio 2018-2025)


In [2]:
# ✅ CELDA 2: PREPROCESAMIENTO (log1p + RobustScaler)
print(f"{TITLE.format(n=2, module='PREPROCESAMIENTO')}")
print(SEP)

features = ml_cfg['features']
X = clustering_df[features].copy()

if ml_cfg['preprocessing']['transform'] == 'log1p':
    X = np.log1p(X)
    print(f"{IND}✅ Transformación: {ml_cfg['preprocessing']['transform']}")

if ml_cfg['preprocessing']['scaler'] == 'RobustScaler':
    scaler = RobustScaler()
    X_scaled_df = pd.DataFrame(scaler.fit_transform(X), columns=features)
    print(f"{IND}✅ Escalado: {ml_cfg['preprocessing']['scaler']}")

print(f"{IND}📊 Matriz lista: {X_scaled_df.shape[0]} municipios × {X_scaled_df.shape[1]} features")
print(f"{IND}🔍 Rango escalado: [{X_scaled_df.min().min():.2f}, {X_scaled_df.max().max():.2f}]")
print(SEP)

✅ CELDA 2 - PREPROCESAMIENTO EJECUTADO
   ✅ Transformación: log1p
   ✅ Escalado: RobustScaler
   📊 Matriz lista: 179 municipios × 4 features
   🔍 Rango escalado: [-7.08, 1.92]


In [3]:
# ✅ CELDA 3: CLUSTERING REVISADO + SELECCIÓN K + NOMBRADO + EXPORTACIÓN
print(f"{TITLE.format(n=3, module='CLUSTERING REVISADO (N=177)')}")
print(SEP)

# 1. Filtrado de códigos DANE inválidos
exclude_cod = [27150, 27493]
mask_clean = ~clustering_df['cod_municipio'].astype(str).isin([str(c) for c in exclude_cod])
clustering_clean = clustering_df[mask_clean].reset_index(drop=True)
X_clean = X_scaled_df[mask_clean].reset_index(drop=True)
assert len(X_clean) == 177, f"⚠️ Error de filtrado: se esperaban 177, se obtuvieron {len(X_clean)}"
print(f"{IND}🗑️ Excluidos: {exclude_cod} (códigos DANE inválidos)")
print(f"{IND}📊 Dataset limpio: {len(X_clean)} municipios × {len(X_clean.columns)} features")

# 2. Selección de k
k_range = [2, 3, 4]
results = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clean)
    sil = silhouette_score(X_clean, labels)
    results.append({'k': k, 'silhouette': sil})
res_df = pd.DataFrame(results)

print(f"\n{IND}📊 Silhouette scores (N=177):")
for _, r in res_df.iterrows():
    print(f"{IND}{PREFIX} k={r['k']}: {r['silhouette']:.4f}")

sorted_sil = res_df.sort_values('silhouette', ascending=False).reset_index(drop=True)
delta = sorted_sil.loc[0,'silhouette'] - sorted_sil.loc[1,'silhouette']
if delta < 0.01:
    k_final = int(sorted_sil.loc[1, 'k']) # Parsimonia
    print(f"{IND}⚖️ Δ < 0.01 → Criterio de parsimonia: k={k_final}")
else:
    k_final = int(sorted_sil.loc[0, 'k'])
    print(f"{IND}✅ Δ ≥ 0.01 → k={k_final}")

# 3. K-Means final
kmeans_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
clustering_clean['cluster_id'] = kmeans_final.fit_predict(X_clean)

# Centroides escala original
centroids_scaled = pd.DataFrame(kmeans_final.cluster_centers_, columns=features)
centroids_orig = np.expm1(scaler.inverse_transform(centroids_scaled))
centroids_orig_df = pd.DataFrame(centroids_orig, columns=features)

# 4. Nombres por severidad
centroids_orig_df['avg'] = centroids_orig_df[features].mean(axis=1)
sorted_ids = centroids_orig_df['avg'].sort_values(ascending=False).index.tolist()
name_map = {sorted_ids[0]: "🔴 Alta severidad", sorted_ids[1]: "🟠 Moderada/Baja"}
clustering_clean['cluster_name'] = clustering_clean['cluster_id'].map(name_map)

print(f"\n{IND}🏷️ Clusters finales (N=177):")
for cid in range(k_final):
    n = (clustering_clean['cluster_id']==cid).sum()
    print(f"{IND}{PREFIX} Cluster {cid} → '{name_map[cid]}' ({n} municipios)")

out_path = PROJECT_ROOT / "data/master" / "tabla_clustering_final.parquet"
clustering_clean.to_parquet(out_path, compression='snappy', index=False)
print(f"\n{IND}💾 Exportado: {out_path}")
print(SEP)
print(f"{IND}✅ Clustering revisado completado.")

✅ CELDA 3 - CLUSTERING REVISADO (N=177) EJECUTADO
   🗑️ Excluidos: [27150, 27493] (códigos DANE inválidos)
   📊 Dataset limpio: 177 municipios × 4 features

   📊 Silhouette scores (N=177):
   📦  k=2.0: 0.4131
   📦  k=3.0: 0.3333
   📦  k=4.0: 0.3279
   ✅ Δ ≥ 0.01 → k=2

   🏷️ Clusters finales (N=177):
   📦  Cluster 0 → '🟠 Moderada/Baja' (70 municipios)
   📦  Cluster 1 → '🔴 Alta severidad' (107 municipios)

   💾 Exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/tabla_clustering_final.parquet
   ✅ Clustering revisado completado.


In [4]:
# ✅ CELDA 4: BENCHMARK + PREDICTOR FINAL (StratifiedKFold + class_weight)
print(f"{TITLE.format(n=4, module='BENCHMARK Y PREDICTOR FINAL')}")
print(SEP)

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score

X = X_clean.copy()
y = clustering_clean['cluster_id']
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM_rbf": SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced'),
    "KNN": KNeighborsClassifier(n_neighbors=5, weights='distance')
}

results = []
print(f"{IND}📊 Benchmark CV F1-Macro (StratifiedKFold=5, class_weight='balanced' donde aplica):")
for name, model in models.items():
    cv_score = cross_val_score(model, X, y, cv=cv_strategy, scoring='f1_macro').mean()
    model.fit(X, y)
    train_f1 = f1_score(y, model.predict(X), average='macro')
    results.append({'model': name, 'cv_f1': cv_score, 'train_f1': train_f1})
    print(f"{IND}{PREFIX} {name}: CV F1={cv_score:.6f} | Train F1={train_f1:.3f}")

results_df = pd.DataFrame(results).sort_values('cv_f1', ascending=False)

# 🔧 SELECCIÓN CON UMBRAL DE EMPATE (tolerancia de 0.001)
best_cv = results_df.iloc[0]['cv_f1']
tied_models = results_df[abs(results_df['cv_f1'] - best_cv) < 0.001]['model'].tolist()

print(f"\n{IND}🔍 Mejor CV F1: {best_cv:.6f}")
print(f"{IND}🔍 Modelos dentro del umbral de empate (±0.001): {tied_models}")

# Si LogisticRegression está en los empatados, seleccionarlo
if 'LogisticRegression' in tied_models:
    best_name = 'LogisticRegression'
    if len(tied_models) > 1:
        print(f"\n{IND}⚖️ Empate técnico entre: {', '.join(tied_models)}")
        print(f"{IND}✅ Criterio de desempate: Interpretabilidad → LogisticRegression")
else:
    best_name = results_df.iloc[0]['model']
    print(f"\n{IND}🏆 Mejor modelo sin empate: {best_name}")

# 🔧 OBTENER EL CV F1 CORRECTO DEL MODELO SELECCIONADO
winner_cv = results_df[results_df['model'] == best_name]['cv_f1'].values[0]

print(f"\n{IND}🏆 Ganador final: {best_name} (CV F1={winner_cv:.6f})")
print(f"{IND}⚠️ Nota: Train F1 es consistencia interna. La métrica real para defensa es CV F1-Macro.")

final_model = models[best_name]
final_model.fit(X, y)

model_path = PROJECT_ROOT / "models" / "final_predictor.pkl"
joblib.dump(final_model, model_path)

meta = {
    "model": best_name,
    "cv_f1_macro": float(winner_cv),
    "features": features,
    "n_municipios": 177,
    "cluster_distribution": clustering_clean['cluster_name'].value_counts().to_dict(),
    "k_final": k_final,
    "validation_strategy": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "class_weight_handling": "balanced (donde soporta el algoritmo)",
    "tie_breaker": "interpretabilidad" if len(tied_models) > 1 else None
}
meta_path = PROJECT_ROOT / "models" / "predictor_metadata.json"
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f"\n{IND}💾 Modelo: {model_path}")
print(f"{IND}💾 Metadatos: {meta_path}")
print(SEP)
print(f"{IND}✅ Predictor final listo. Métrica honesta y defendible: CV F1-Macro = {winner_cv:.6f}")

✅ CELDA 4 - BENCHMARK Y PREDICTOR FINAL EJECUTADO
   📊 Benchmark CV F1-Macro (StratifiedKFold=5, class_weight='balanced' donde aplica):
   📦  LogisticRegression: CV F1=0.976905 | Train F1=0.994
   📦  RandomForest: CV F1=0.923423 | Train F1=1.000
   📦  GradientBoosting: CV F1=0.923080 | Train F1=1.000
   📦  SVM_rbf: CV F1=0.976910 | Train F1=0.994
   📦  DecisionTree: CV F1=0.881709 | Train F1=1.000
   📦  KNN: CV F1=0.952430 | Train F1=1.000

   🔍 Mejor CV F1: 0.976910
   🔍 Modelos dentro del umbral de empate (±0.001): ['SVM_rbf', 'LogisticRegression']

   ⚖️ Empate técnico entre: SVM_rbf, LogisticRegression
   ✅ Criterio de desempate: Interpretabilidad → LogisticRegression

   🏆 Ganador final: LogisticRegression (CV F1=0.976905)
   ⚠️ Nota: Train F1 es consistencia interna. La métrica real para defensa es CV F1-Macro.

   💾 Modelo: /Users/anaaguirre/Documents/Cicatrices_invisibles/models/final_predictor.pkl
   💾 Metadatos: /Users/anaaguirre/Documents/Cicatrices_invisibles/models/predict

In [5]:
# ✅ CELDA 5: VALIDACIÓN ROBUSTA + ARI + COHERENCIA ICV + FRONTERA (RF-1, RF-2, O-B)
from sklearn.metrics import adjusted_rand_score, silhouette_samples
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f"{TITLE.format(n=5, module='VALIDACIÓN ROBUSTA Y COHERENCIA')}")
print(SEP)

# 1. RF-1 / O-C: Validación con Coeficiente de Variación (sin umbral arbitrario)
centroids_orig = np.expm1(scaler.inverse_transform(kmeans_final.cluster_centers_))
centroids_df = pd.DataFrame(centroids_orig, columns=features)
centroids_norm = centroids_df.div(centroids_df.sum(axis=1), axis=0)

cv_magnitud = (centroids_df.std() / centroids_df.mean()).mean()
cv_proporcion = (centroids_norm.std() / centroids_norm.mean()).mean()
ratio_cv = cv_magnitud / cv_proporcion

print(f"{IND}🔍 RF-1 | Validación sustantiva (CV comparable):")
print(f"{IND}{PREFIX} CV proporciones (estructura): {cv_proporcion:.3f}")
print(f"{IND}{PREFIX} CV magnitudes (severidad): {cv_magnitud:.3f}")
print(f"{IND}{PREFIX} Ratio CV: {ratio_cv:.2f}x")
print(f"{IND}⚠️ Limitación: CV calculado sobre 2 centroides. Indicativo, no prueba robusta.")
print(f"{IND}✅ Interpretación: Variabilidad dominada por magnitud (ratio {ratio_cv:.1f}x) → gradiente continuo discretizado.")

# 2. O-B: Municipios frontera (Silhouette individual ≤ 0)
sil_individual = silhouette_samples(X_clean, clustering_clean['cluster_id'])
border_muni = (sil_individual <= 0).sum()
pct_border = border_muni / len(clustering_clean) * 100

print(f"\n{IND}📍 O-B | Municipios en zona de solapamiento (Silhouette ≤ 0): {border_muni} ({pct_border:.1f}%)")
if pct_border > 20:
    print(f"{IND}⚠️ Dashboard: Marcar con disclaimer de asignación frágil.")
else:
    print(f"{IND}✅ Asignación robusta para la mayoría.")

# 3. RF-2: Bootstrap ARI (k=2, N=179)
print(f"\n{IND}🔄 RF-2 | Bootstrap ARI (modelo actual k=2, N={len(clustering_clean)})...")
n_boot, ari_scores, n_sub = 200, [], int(len(clustering_clean)*0.8)
np.random.seed(42)
for _ in range(n_boot):
    idx = np.random.choice(len(clustering_clean), n_sub, replace=False)
    k_boot = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels_boot = k_boot.fit_predict(X_clean.iloc[idx])
    if len(np.unique(labels_boot)) == 2:
        ari_scores.append(adjusted_rand_score(clustering_clean.iloc[idx]['cluster_id'], labels_boot))

ari_mean = np.mean(ari_scores) if ari_scores else 0
print(f"{IND}{PREFIX} ARI promedio: {ari_mean:.3f} | Umbral >0.80 → {'✅ Estable' if ari_mean > 0.80 else '⚠️ Moderado'}")

print(SEP)
print(f"{IND}✅ Validación robusta completada. Métricas trazables y defensibles.")

✅ CELDA 5 - VALIDACIÓN ROBUSTA Y COHERENCIA EJECUTADO
   🔍 RF-1 | Validación sustantiva (CV comparable):
   📦  CV proporciones (estructura): 0.327
   📦  CV magnitudes (severidad): 0.790
   📦  Ratio CV: 2.42x
   ⚠️ Limitación: CV calculado sobre 2 centroides. Indicativo, no prueba robusta.
   ✅ Interpretación: Variabilidad dominada por magnitud (ratio 2.4x) → gradiente continuo discretizado.

   📍 O-B | Municipios en zona de solapamiento (Silhouette ≤ 0): 11 (6.2%)
   ✅ Asignación robusta para la mayoría.

   🔄 RF-2 | Bootstrap ARI (modelo actual k=2, N=177)...
   📦  ARI promedio: 0.967 | Umbral >0.80 → ✅ Estable
   ✅ Validación robusta completada. Métricas trazables y defensibles.


In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
from scipy.optimize import linear_sum_assignment
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

# Cargar configuración de display
with open(PROJECT_ROOT / "config" / "base_config.json", 'r', encoding='utf-8') as f:
    base_cfg = json.load(f)

disp = base_cfg.get('display_config', {})
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO').format(n=6, module="VALIDACIÓN TEMPORAL")

# ============================================================
# 🔍 VALIDACIÓN TEMPORAL (temporal_split)
# Propósito: verificar que los clusters son ESTABLES en el tiempo
# Metodología: entrenar K-Means en 2018-2022 vs 2023-2025
#              y comparar asignaciones con ARI
# ============================================================

print(f"\n{TITLE}")
print(SEP)

# 1. Cargar configuración ML
with open(PROJECT_ROOT / "config" / "ml_config.json", 'r', encoding='utf-8') as f:
    ml_config = json.load(f)

features = ml_config['features']
train_years = ml_config['validation']['temporal_split']['train']
test_years = ml_config['validation']['temporal_split']['test']
stability_threshold = ml_config['validation']['temporal_split']['stability_threshold']
random_state = ml_config['validation']['random_state']

print(f"{IND}📅 Período train: {train_years}")
print(f"{IND}📅 Período test:  {test_years}")
print(f"{IND}🎯 Umbral estabilidad: ARI > {stability_threshold}")
print(f"{IND}🔢 Features: {features}")

# 2. Cargar master_table
master_path = PROJECT_ROOT / "data" / "master" / "master_table.parquet"
master_df = pd.read_parquet(master_path)
print(f"\n{IND}{PREFIX}master_table cargado: {len(master_df):,} filas × {len(master_df.columns)} cols")

# 3. Excluir municipios con subregistro
municipios_excluidos = ['27150', '27493']
master_df = master_df[~master_df['cod_municipio'].astype(str).isin(municipios_excluidos)]
print(f"{IND}{PREFIX}Municipios excluidos (subregistro): {municipios_excluidos}")
print(f"{IND}{PREFIX}Dataset limpio: {master_df['cod_municipio'].nunique()} municipios únicos")

# 4. Parsear rangos
train_start, train_end = map(int, train_years.split('-'))
test_start, test_end = map(int, test_years.split('-'))

# 5. Función: preprocessing consistente
def preparar_datos_temporales(df, train_range, test_range, features):
    """Prepara datos con preprocessing CONSISTENTE."""
    df_train = df[(df['anio_hecho'] >= train_range[0]) & (df['anio_hecho'] <= train_range[1])].copy()
    df_test = df[(df['anio_hecho'] >= test_range[0]) & (df['anio_hecho'] <= test_range[1])].copy()

    X_train_raw = df_train.groupby('cod_municipio')[features].mean()
    X_test_raw = df_test.groupby('cod_municipio')[features].mean()

    municipios_comunes = X_train_raw.index.intersection(X_test_raw.index)
    X_train_raw = X_train_raw.loc[municipios_comunes]
    X_test_raw = X_test_raw.loc[municipios_comunes]

    X_train_log = np.log1p(X_train_raw)
    X_test_log = np.log1p(X_test_raw)

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_log)
    X_test_scaled = scaler.transform(X_test_log)

    return X_train_scaled, X_test_scaled, municipios_comunes

# 6. Preparar datos
print(f"\n{IND}🔄 Preparando datos con preprocessing consistente...")
X_train, X_test, municipios_comunes = preparar_datos_temporales(
    master_df,
    (train_start, train_end),
    (test_start, test_end),
    features
)

print(f"{IND}{PREFIX}Train: {len(X_train)} municipios")
print(f"{IND}{PREFIX}Test:  {len(X_test)} municipios")
print(f"{IND}{PREFIX}Comunes: {len(municipios_comunes)}")

# 7. Entrenar K-Means
print(f"\n{IND}🤖 Entrenando K-Means (k=2)...")
kmeans_train = KMeans(n_clusters=2, random_state=random_state, n_init=10)
kmeans_test = KMeans(n_clusters=2, random_state=random_state, n_init=10)

labels_train = kmeans_train.fit_predict(X_train)
labels_test = kmeans_test.fit_predict(X_test)

# 8. Label matching con algoritmo húngaro
def match_labels(labels_train, labels_test):
    """Encuentra la mejor correspondencia entre etiquetas."""
    cm = confusion_matrix(labels_train, labels_test)
    row_ind, col_ind = linear_sum_assignment(-cm)
    label_map = {old: new for old, new in zip(col_ind, row_ind)}
    labels_test_matched = np.array([label_map[l] for l in labels_test])
    return labels_test_matched, cm

labels_test_matched, cm = match_labels(labels_train, labels_test)

# 9. Calcular ARI con etiquetas corregidas
ari_score = adjusted_rand_score(labels_train, labels_test_matched)

print(f"\n{IND}📊 RESULTADOS (con label matching):")
print(f"{IND}{PREFIX}ARI entre períodos: {ari_score:.4f}")
print(f"{IND}{PREFIX}Umbral declarado:   >{stability_threshold}")
estado = "✅ ESTABLE" if ari_score > stability_threshold else "❌ INESTABLE"
print(f"{IND}{PREFIX}Estado: {estado}")

# 10. Distribución
print(f"\n{IND}📈 Distribución de clusters (después de label matching):")
unique_train, counts_train = np.unique(labels_train, return_counts=True)
unique_test, counts_test = np.unique(labels_test_matched, return_counts=True)

print(f"{IND}{PREFIX}{train_years}:")
for cluster, count in zip(unique_train, counts_train):
    print(f"{IND}{IND}• Cluster {cluster}: {count} municipios ({count/len(labels_train)*100:.1f}%)")

print(f"{IND}{PREFIX}{test_years}:")
for cluster, count in zip(unique_test, counts_test):
    print(f"{IND}{IND}• Cluster {cluster}: {count} municipios ({count/len(labels_test)*100:.1f}%)")

# 11. Matriz de confusión
print(f"\n{IND}🔀 Matriz de confusión temporal (después de label matching):")
cm_matched = confusion_matrix(labels_train, labels_test_matched)
print(f"{IND}{IND}             Test (matched)")
print(f"{IND}{IND}          C0   C1")
print(f"{IND}{IND}Train C0 [{cm_matched[0][0]:3d}  {cm_matched[0][1]:3d}]")
print(f"{IND}{IND}      C1 [{cm_matched[1][0]:3d}  {cm_matched[1][1]:3d}]")
print(f"{IND}{PREFIX}Municipios que cambiaron: {cm_matched[0][1] + cm_matched[1][0]}")

# 12. Guardar evidencia
evidencia = {
    "temporal_split": {
        "version": "v2_corregida",
        "train_period": train_years,
        "test_period": test_years,
        "municipios_comunes": int(len(municipios_comunes)),
        "ari_score": float(ari_score),
        "stability_threshold": float(stability_threshold),
        "status": "ESTABLE" if ari_score > stability_threshold else "INESTABLE",
        "label_matching_aplicado": True,
        "distribucion_train": {int(k): int(v) for k, v in zip(unique_train, counts_train)},
        "distribucion_test": {int(k): int(v) for k, v in zip(unique_test, counts_test)},
        "matriz_confusion_matched": cm_matched.tolist(),
        "municipios_cambiaron": int(cm_matched[0][1] + cm_matched[1][0])
    }
}

evidencia_path = PROJECT_ROOT / "docs" / "evidencia_temporal_split.json"
with open(evidencia_path, 'w', encoding='utf-8') as f:
    json.dump(evidencia, f, indent=2, ensure_ascii=False)

print(f"\n{IND}💾 Evidencia guardada: {evidencia_path}")
print(SEP)
print(f"✅ Etapa 6 completada. Validación temporal ejecutada.")
print(SEP)


✅ CELDA 6 - VALIDACIÓN TEMPORAL EJECUTADO
   📅 Período train: 2018-2022
   📅 Período test:  2023-2025
   🎯 Umbral estabilidad: ARI > 0.3
   🔢 Features: ['tasa_vif_nna_f', 'tasa_sexual_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_adultas_f']

   📦 master_table cargado: 1,432 filas × 31 cols
   📦 Municipios excluidos (subregistro): ['27150', '27493']
   📦 Dataset limpio: 177 municipios únicos

   🔄 Preparando datos con preprocessing consistente...
   📦 Train: 177 municipios
   📦 Test:  177 municipios
   📦 Comunes: 177

   🤖 Entrenando K-Means (k=2)...

   📊 RESULTADOS (con label matching):
   📦 ARI entre períodos: 0.3268
   📦 Umbral declarado:   >0.3
   📦 Estado: ✅ ESTABLE

   📈 Distribución de clusters (después de label matching):
   📦 2018-2022:
      • Cluster 0: 120 municipios (67.8%)
      • Cluster 1: 57 municipios (32.2%)
   📦 2023-2025:
      • Cluster 0: 117 municipios (66.1%)
      • Cluster 1: 60 municipios (33.9%)

   🔀 Matriz de confusión temporal (después de label matching):

In [7]:
# ============================================================================
# 🔐 CELDA V1: EVIDENCIA RF-C - EXCLUSIÓN DE MUNICIPIOS (NO BORRAR)
# Propósito: Demostrar conteos absolutos que justifican exclusión de 27150, 27493
# Revisión: RF-C | Fecha: [FECHA_ACTUAL]
# ============================================================================
import pandas as pd
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

exclude = ['27150', '27493']
cols = ['cod_municipio', 'anio_hecho', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f']

master = pd.read_parquet(PROJECT_ROOT / "data/master" / "master_table.parquet", columns=cols)
verif = master[master['cod_municipio'].astype(str).str.strip().isin(exclude)].sort_values(['cod_municipio','anio_hecho'])

print("📊 CONTEOS ABSOLUTOS DS1/DS2 - Municipios excluidos (Evidencia RF-C):")
print(verif.to_string(index=False))
print(f"\n📌 Conclusión trazable: VIF=0 sostenido 8 años; sexual=1 caso en 2018 luego 0 → asimetría institucional de registro, no ausencia de violencia.")

📊 CONTEOS ABSOLUTOS DS1/DS2 - Municipios excluidos (Evidencia RF-C):
cod_municipio  anio_hecho  casos_vif_nna_f  casos_vif_adultas_f  casos_sexual_nna_f  casos_sexual_adultas_f
        27150        2018                0                    0                   1                       1
        27150        2019                0                    0                   0                       0
        27150        2020                0                    0                   0                       0
        27150        2021                0                    0                   0                       0
        27150        2022                0                    0                   0                       0
        27150        2023                0                    0                   0                       0
        27150        2024                0                    0                   0                       0
        27150        2025                0                    0    

In [8]:
# ============================================================================
# 🔐 CELDA V2: EVIDENCIA O-A/RF-A - CONCORDANCIA ICV vs CLUSTER (NO BORRAR)
# Propósito: Demostrar % de alineación que justifica narrativa de "discretización"
# Granularidad: MUNICIPIO (N=177 exactos)
# Revisión: RF-A, O-A | Fecha: [FECHA_ACTUAL]
# ============================================================================
import pandas as pd
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

clust_path = PROJECT_ROOT / "data/master" / "tabla_clustering_final.parquet"
master_path = PROJECT_ROOT / "data/master" / "master_table.parquet"

print("1️⃣ Cargando datos...")
clust = pd.read_parquet(clust_path)  # 177 municipios (excluye 27150, 27493)
master = pd.read_parquet(master_path)  # 179 municipios originales × años

# Detectar columna ICV
icv_candidates = [c for c in master.columns if 'icv' in c.lower()]
if not icv_candidates:
    raise ValueError("❌ No se encontró columna ICV en master_table.parquet")
icv_original = icv_candidates[0]
print(f"   ✅ Columna ICV detectada: '{icv_original}'")

# 🛡️ FIX 1: Filtrar master para que SOLO tenga los 177 municipios del clustering
# Esto evita que municipios excluidos (27150, 27493) contaminen el análisis
cod_muni_validos = clust['cod_municipio'].astype(str).str.strip().unique()
master_filtered = master[master['cod_municipio'].astype(str).str.strip().isin(cod_muni_validos)].copy()
print(f"   ✅ Master filtrado: {master_filtered['cod_municipio'].nunique()} municipios únicos")

# 🛡️ FIX 2: Agrupar por municipio para tener 1 valor ICV × 1 municipio (promedio 2018-2025)
master_muni = master_filtered.groupby('cod_municipio')[icv_original].mean().reset_index()
master_muni = master_muni.rename(columns={icv_original: 'icv_val'})
print(f"   ✅ ICV agregado: {len(master_muni)} municipios × 1 fila")

# Normalizar tipos para merge seguro
clust['cod_municipio'] = clust['cod_municipio'].astype(str).str.strip()
master_muni['cod_municipio'] = master_muni['cod_municipio'].astype(str).str.strip()

print("2️⃣ Ejecutando merge 1:1 (municipio ↔ municipio)...")
df = clust.merge(master_muni, on='cod_municipio', how='inner').dropna(subset=['icv_val'])
print(f"   ✅ Registros finales: {len(df)} municipios (debe ser exactamente 177)")

# Validación de seguridad
if len(df) != 177:
    print(f"   ⚠️ ALERTA: Se esperaban 177, se obtuvieron {len(df)}. Revisar filtrado.")

print("3️⃣ Calculando concordancia a nivel municipio...")
df['tercil'] = pd.qcut(df['icv_val'], q=3, labels=['Bajo','Medio','Alto'], duplicates='drop')
tabla = pd.crosstab(df['cluster_name'], df['tercil'], normalize='columns')
print("\n📊 CONCORDANCIA CLUSTER vs ICV (Nivel MUNICIPIO N=177 - Evidencia RF-A/O-A):")
print(tabla.round(2))

desacuerdo = len(df[
    (df['cluster_name'].str.contains('Alta')) & (df['tercil'].isin(['Bajo','Medio'])) |
    (df['cluster_name'].str.contains('Moderada')) & (df['tercil'].isin(['Alto']))
])
total = len(df)
print(f"\n⚠️ Municipios en desacuerdo: {desacuerdo} de {total} ({desacuerdo/total*100:.1f}%)")
print(f"📌 Conclusión trazable: {100-(desacuerdo/total*100):.1f}% alineación → cluster discretiza continuo ICV a nivel municipal.")

1️⃣ Cargando datos...
   ✅ Columna ICV detectada: 'icv_gen_f_score'
   ✅ Master filtrado: 177 municipios únicos
   ✅ ICV agregado: 177 municipios × 1 fila
2️⃣ Ejecutando merge 1:1 (municipio ↔ municipio)...
   ✅ Registros finales: 177 municipios (debe ser exactamente 177)
3️⃣ Calculando concordancia a nivel municipio...

📊 CONCORDANCIA CLUSTER vs ICV (Nivel MUNICIPIO N=177 - Evidencia RF-A/O-A):
tercil            Bajo  Medio  Alto
cluster_name                       
🔴 Alta severidad   0.0   0.81   1.0
🟠 Moderada/Baja    1.0   0.19   0.0

⚠️ Municipios en desacuerdo: 48 de 177 (27.1%)
📌 Conclusión trazable: 72.9% alineación → cluster discretiza continuo ICV a nivel municipal.
